<a href="https://colab.research.google.com/github/vasilyryabtsev/futures-price-prediction/blob/vasily/twitter/ML_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42

In [2]:
df = pd.read_csv('https://github.com/vasilyryabtsev/futures-price-prediction/blob/vasily/twitter/tweets.csv?raw=true').drop('Unnamed: 0', axis=1)
df

,text,is_quote_status,view_count,has_card,urls,ticker,day,month,year,is_in_reply_to,is_view_count,1_day_after
0,The chance of $MSFT winning an appeal of the ...,0,113387.0,0,0,MSFT,26,4,2023,0,1,1
1,We love and appreciate all the volunteers at t...,0,707.0,0,0,NVDA,6,5,2024,0,1,1
2,Today walking the lab on the NJ beach - tomorr...,0,702.0,0,0,META,8,5,2024,0,1,1
3,Today walking the lab on the NJ beach - tomorr...,0,702.0,0,0,NVDA,8,5,2024,0,1,0
4,Good Morning from a dog walk on the $NVDA resc...,0,934.0,0,0,NVDA,5,5,2024,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
522,Autonomy takes center stage in this quarter's ...,0,6220.0,1,0,TSLA,25,10,2024,0,1,1
523,Had to eat McDonalds over Wendy's this morning...,0,18324.0,0,0,MCD,23,9,2024,0,1,1
524,$BA big equity raise,0,1021.0,0,0,BA,28,10,2024,0,1,0
525,$JPM gives the BTC miners 9 months to get a de...,0,1071.0,1,1,JPM,24,10,2024,0,1,1


# Текстовые признаки

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df['text'].to_frame(), df['1_day_after'], test_size=0.1, random_state=RANDOM_STATE)

## Bag Of Words

In [ ]:
vec = CountVectorizer(ngram_range=(1, 1), stop_words='english')

X_train = vec.fit_transform(X_train['text'])
X_test = vec.transform(X_test['text'])

scaler = MaxAbsScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf = LogisticRegression(max_iter=200, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

pred_train = clf.predict(X_train)
pred_test = clf.predict(X_test)

print('accuracy (train, test):', accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test))
print('roc-auc (train, test):', roc_auc_score(y_train, pred_train), roc_auc_score(y_test, pred_test))

Переобучение. Изменение ngram_range проблему не решило.

In [ ]:
print(classification_report(y_test, pred_test))

## Tf-idf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(df0['text'].to_frame(), df0['1_day_after'], test_size=0.1, random_state=RANDOM_STATE)

vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 1)) # ngram_range = (1,1) - векторизуем только отдельные слова

X_train = vec.fit_transform(X_train['text'])
X_test = vec.transform(X_test['text'])

scaler = MaxAbsScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf = LogisticRegression(max_iter=200, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

pred_train = clf.predict(X_train)
pred_test = clf.predict(X_test)

print('accuracy (train, test):', accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test))
print('roc-auc (train, test):', roc_auc_score(y_train, pred_train), roc_auc_score(y_test, pred_test))

Переобучение. Изменение ngram_range проблему не решило.

In [ ]:
print(classification_report(y_test, pred_test))

## [finbert-tone](https://huggingface.co/yiyanghkust/finbert-tone?text=growth+is+strong+and+we+have+plenty+of+liquidity)

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline

finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone',num_labels=3)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')

nlp = pipeline("sentiment-analysis", model=finbert, tokenizer=tokenizer)

sentences = "there is a shortage of capital, and we need extra financing"
results = nlp(sentences)
print(results)  #LABEL_0: neutral; LABEL_1: positive; LABEL_2: negative

In [ ]:
class FinBertTransformer(BaseEstimator, TransformerMixin):
    '''
    Добавляет в датасет два новых признака, полученных из text:
    label - оценка текста Negative/Positive/Neutral
    score - уверенность в оценке 0.0 - 1.0 float
    '''
    def __init__(self, pipe):
        self.pipe = pipe

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy['text'] = X_copy['text'].apply(self.pipe)
        X_copy['label'] = X_copy['text'].apply(lambda x: x[0]['label'])
        X_copy['score'] = X_copy['text'].apply(lambda x: x[0]['score'])

        return X_copy.drop('text', axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df0['text'].to_frame(), df0['1_day_after'], test_size=0.1, random_state=RANDOM_STATE)

nlp = pipeline("sentiment-analysis", model=finbert, tokenizer=tokenizer)
finbert_score = FinBertTransformer(nlp)

X_train = finbert_score.fit_transform(X_train)
X_test = finbert_score.transform(X_test)

X_train = pd.get_dummies(X_train, 'label', drop_first=True)
X_test = pd.get_dummies(X_test, 'label', drop_first=True)

clf = LogisticRegression(max_iter=200, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

pred_train = clf.predict(X_train)
pred_test = clf.predict(X_test)

print('accuracy (train, test):', accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test))
print('roc-auc (train, test):', roc_auc_score(y_train, pred_train), roc_auc_score(y_test, pred_test))

Переобучения нет!

In [ ]:
print(classification_report(y_test, pred_test))

## [finbert ProcusAI](https://huggingface.co/ProsusAI/finbert)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df0['text'].to_frame(), df0['1_day_after'], test_size=0.1, random_state=RANDOM_STATE)

nlp = pipeline("text-classification", model="ProsusAI/finbert")
finbert_score = FinBertTransformer(nlp)

X_train = finbert_score.fit_transform(X_train)
X_test = finbert_score.transform(X_test)

X_train = pd.get_dummies(X_train, 'label', drop_first=True)
X_test = pd.get_dummies(X_test, 'label', drop_first=True)

clf = LogisticRegression(max_iter=200, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

pred_train = clf.predict(X_train)
pred_test = clf.predict(X_test)

print('accuracy (train, test):', accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test))
print('roc-auc (train, test):', roc_auc_score(y_train, pred_train), roc_auc_score(y_test, pred_test))

Небольшое переобучение

In [ ]:
print(classification_report(y_test, pred_test))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import torch
from sklearn.metrics import accuracy_score, roc_auc_score

# Загрузка модели и токенизатора
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

# Токенезация
tokenized = df0['text'].apply((lambda x: tokenizer.encode(x, add_special_tokens=True)))

# Паддинг (чтобы все тексты были одинаковой длины)
max_len = 0
for i in tokenized.values:
    if len(i) > max_len:
        max_len = len(i)
padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])

# Masking (нужно показать модели, что все нули это пустое место)
attention_mask = np.where(padded != 0, 1, 0)
input_ids = torch.tensor(padded)
attention_mask = torch.tensor(attention_mask)

# Применение модели
with torch.no_grad():
    last_hidden_states = model(input_ids, attention_mask=attention_mask)

features = last_hidden_states[0].numpy()

X_train, X_test, y_train, y_test = train_test_split(features, df0['1_day_after'], test_size=0.1, random_state=RANDOM_STATE)

clf = LogisticRegression()
clf.fit(X_train, y_train)

pred_train = clf.predict(X_train)
pred_test = clf.predict(X_test)

print('accuracy (train, test):', accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test))
print('roc-auc (train, test):', roc_auc_score(y_train, pred_train), roc_auc_score(y_test, pred_test))

In [32]:
X_train, X_test, y_train, y_test = train_test_split(df0, df0['1_day_after'], test_size=0.1, random_state=RANDOM_STATE)

In [24]:
text_df = pd.DataFrame(data={'text': ['BUY $APPL']})

In [29]:
from sklearn.preprocessing import FunctionTransformer

def preprocessing(X, y=None):
    X_copy = X.copy()

    tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

    # Токенезация
    tokenized = X_copy['text'].apply((lambda x: tokenizer.encode(x, add_special_tokens=True)))

    # Паддинг (чтобы все тексты были одинаковой длины)
    max_len = 0
    for i in tokenized.values:
        if len(i) > max_len:
            max_len = len(i)
    padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])

    # Masking (нужно показать модели, что все нули это пустое место)
    attention_mask = np.where(padded != 0, 1, 0)
    input_ids = torch.tensor(padded)
    attention_mask = torch.tensor(attention_mask)

    # Применение модели
    with torch.no_grad():
        last_hidden_states = model(input_ids, attention_mask=attention_mask)

    features = last_hidden_states[0].numpy()

    return features

finbert = FunctionTransformer(preprocessing)

from sklearn.pipeline import Pipeline

pipe = Pipeline(steps=[
    ('procus_ai', finbert),
    ('clf', LogisticRegression())
])

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
pipe.predict(text_df)

In [ ]:
print(classification_report(y_test, pred_test))

In [ ]:
def preprocessing(X, y=None):
    X_copy = X.copy()


## Word2vec

In [ ]:
from gensim.models import Word2Vec

sent = [row.split() for row in df0['text']]

HIDDEN = 100 # каждое слово закодированно числовым вектором длины 100

model = Word2Vec(min_count=20,
                     window=2,
                     vector_size=HIDDEN,
                     sample=6e-5,
                     alpha=0.03,
                     min_alpha=0.0007,
                     negative=20,
                     workers=2)
model.build_vocab(sent, progress_per=10000) # собираем словарь слов
model.train(sent, total_examples=model.corpus_count, epochs=30, report_delay=1)

In [ ]:
def get_mean_w2v_vector(sentence):
    Sum = 0
    Count = 0

    try:
      words = sentence.split()
    except TypeError:
      words = []

    for w in words:
        if w in model.wv:
            Sum += model.wv[w]
            # Sum += glove_vectors[w]
            Count += 1

    if Count == 0:
        return 0

    return Sum / Count

In [ ]:
NewCols = ['col'+str(i) for i in range(HIDDEN)]

X_train, X_test, y_train, y_test = train_test_split(df0['text'].to_frame(), df0['1_day_after'], test_size=0.1, random_state=RANDOM_STATE)

X_train['vectors'] = X_train.map(get_mean_w2v_vector)
X_test['vectors'] = X_test.map(get_mean_w2v_vector)

X_train.head()

In [ ]:
IdxTrain = []

for ix, row in X_train.iterrows():
    if not isinstance(row['vectors'],np.ndarray):
        IdxTrain.append(ix)

IdxTest = []

for ix, row in X_test.iterrows():
    if not isinstance(row['vectors'],np.ndarray):
        IdxTest.append(ix)

In [ ]:
X_train.drop(index=IdxTrain, inplace=True)
X_test.drop(index=IdxTest, inplace=True)

y_train = y_train.drop(index=IdxTrain)
y_test = y_test.drop(index=IdxTest)

In [ ]:
X_train[NewCols] = pd.DataFrame(X_train['vectors'].tolist(), index=X_train.index)
X_test[NewCols] = pd.DataFrame(X_test['vectors'].tolist(), index=X_test.index)

In [ ]:
X_train.head()

In [ ]:
X_train.drop(['text','vectors'], axis=1, inplace=True)
X_test.drop(['text','vectors'], axis=1, inplace=True)

In [ ]:
clf = LogisticRegression()
clf.fit(X_train, y_train)

pred_train = clf.predict(X_train)
pred_test = clf.predict(X_test)

print('accuracy (train, test):', accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test))
print('roc-auc (train, test):', roc_auc_score(y_train, pred_train), roc_auc_score(y_test, pred_test))

Переобучения нет, но качество очень низкое

## ProcusAI + Численные признаки

In [ ]:
numeric = np.array(df0.drop(['1_day_after', '3_day_after', '7_day_after'], axis=1).select_dtypes(include='number'))

In [ ]:
data = np.hstack([numeric, features])

In [ ]:
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(data, df0['1_day_after'], test_size=0.1, random_state=RANDOM_STATE)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train, y_train)

pred_train = clf.predict(X_train)
pred_test = clf.predict(X_test)


print('accuracy (train, test):', accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test))
print('roc-auc (train, test):', roc_auc_score(y_train, pred_train), roc_auc_score(y_test, pred_test))

In [ ]:
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(LogisticRegression(), {'C': [0.01 * k for k in range(1, 201)]}, cv=3)

gs.fit(X_train, y_train)

print(f'{gs.best_params_=}')

In [ ]:
clf = LogisticRegression(C=1.77)
clf.fit(X_train, y_train)

pred_train = clf.predict(X_train)
pred_test = clf.predict(X_test)


print('accuracy (train, test):', accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test))
print('roc-auc (train, test):', roc_auc_score(y_train, pred_train), roc_auc_score(y_test, pred_test))

In [ ]:
print(classification_report(y_test, pred_test))

## Выводы

Наилучший результат показала логистическая регрессия, обученная на текстах, векторизованных с помощью предобученной finbert (ProcusAI), и на вещественных признаках. Так же для данной модели был осуществлен подбор оптимального гиперпараметра: C=1.77.

accuracy (train, test): 0.611, 0.679

roc-auc (train, test): 0.605, 0.653

По значению метрик видно, что модель скорее всего немного недообучилась. Это связано с маленьким размером датасета.

## Что можно улучшить?

1. Собрать больше данных
2. Попробовать SVM на разных ядрах